# End-to-End Data Analysis Case Study

**Course:** Data Science  
**Lesson:** 08  
**Difficulty:** Intermediate  
**Project Type:** Integrated Case Study

## Project Objective

Apply the complete course workflow to analyze workforce patterns involving salary, experience, performance, training, satisfaction, absence, department, education, and gender.

## Learning Outcomes

- Define analytical questions.
- Inspect and validate a dataset.
- Clean and prepare variables.
- Create descriptive summaries and charts.
- Analyze correlations and grouped patterns.
- Perform appropriate hypothesis tests.
- Interpret practical significance.
- Communicate findings and limitations.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

## 1. Analytical Questions

1. What are the main workforce characteristics?
2. Which departments have the highest salaries and performance?
3. How are experience, salary, training, and performance related?
4. Does salary differ between two selected gender groups?
5. Does salary differ across departments?
6. Is education level associated with department?
7. Which findings support useful recommendations?

## 2. Load the Dataset

In [ ]:
candidate_paths=[
    Path("../datasets/employee_statistics.csv"),
    Path("Data-Science-Course/datasets/employee_statistics.csv"),
    Path("datasets/employee_statistics.csv"),
]
data_path=next((p for p in candidate_paths if p.exists()),None)
if data_path is None:
    raise FileNotFoundError("Place employee_statistics.csv in Data-Science-Course/datasets/.")
df=pd.read_csv(data_path,parse_dates=["Hire_Date"])
df.head()

In [ ]:
df.shape

## 3. Initial Inspection

In [ ]:
df.info()

In [ ]:
df.sample(5,random_state=42)

In [ ]:
df.columns.tolist()

## 4. Data Quality Assessment

In [ ]:
quality_report=pd.DataFrame({
    "Data_Type":df.dtypes.astype(str),
    "Missing_Values":df.isna().sum(),
    "Missing_Percentage":(df.isna().mean()*100).round(2),
    "Unique_Values":df.nunique()
})
quality_report

In [ ]:
pd.Series({
    "Duplicate Rows":df.duplicated().sum(),
    "Negative Ages":(df["Age"]<0).sum(),
    "Negative Experience":(df["Years_Experience"]<0).sum(),
    "Negative Salary":(df["Annual_Salary_USD"]<0).sum(),
    "Negative Weekly Hours":(df["Weekly_Hours"]<0).sum(),
    "Negative Absence Days":(df["Absence_Days"]<0).sum()
})

## 5. Create a Clean Analysis Copy

In [ ]:
analysis_df=df.copy().drop_duplicates()

for column in analysis_df.select_dtypes(include="number").columns:
    if analysis_df[column].isna().any():
        analysis_df[column]=analysis_df[column].fillna(analysis_df[column].median())

for column in analysis_df.select_dtypes(include="object").columns:
    if analysis_df[column].isna().any():
        analysis_df[column]=analysis_df[column].fillna(analysis_df[column].mode().iloc[0])

analysis_df.isna().sum().sum()

The original dataset remains unchanged. All later work uses `analysis_df`.

## 6. Create Derived Variables

In [ ]:
analysis_df["Experience_Level"]=pd.cut(
    analysis_df["Years_Experience"],
    bins=[-np.inf,3,8,15,np.inf],
    labels=["Entry","Developing","Experienced","Senior"]
)
analysis_df["Salary_Band"]=pd.qcut(
    analysis_df["Annual_Salary_USD"],
    q=4,
    labels=["Low","Lower-Middle","Upper-Middle","High"],
    duplicates="drop"
)
analysis_df["Hire_Year"]=analysis_df["Hire_Date"].dt.year
analysis_df[["Years_Experience","Experience_Level","Annual_Salary_USD","Salary_Band","Hire_Year"]].head()

## 7. Descriptive Statistics

In [ ]:
key_columns=[
    "Age","Years_Experience","Annual_Salary_USD","Weekly_Hours",
    "Performance_Score","Job_Satisfaction","Training_Hours","Absence_Days"
]
analysis_df[key_columns].describe().T.round(2)

## 8. Workforce Composition

In [ ]:
analysis_df["Department"].value_counts().rename("Employee_Count").to_frame()

In [ ]:
analysis_df["Education_Level"].value_counts().rename("Employee_Count").to_frame()

## 9. Salary Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data=analysis_df,x="Annual_Salary_USD",bins=25,kde=True)
plt.title("Annual Salary Distribution")
plt.xlabel("Annual Salary (USD)")
plt.tight_layout()
plt.show()

## 10. Salary Outlier Review

In [ ]:
q1=analysis_df["Annual_Salary_USD"].quantile(0.25)
q3=analysis_df["Annual_Salary_USD"].quantile(0.75)
iqr=q3-q1
lower=q1-1.5*iqr
upper=q3+1.5*iqr
salary_outliers=analysis_df[
    (analysis_df["Annual_Salary_USD"]<lower)|
    (analysis_df["Annual_Salary_USD"]>upper)
]
pd.Series({"Q1":q1,"Q3":q3,"IQR":iqr,"Lower Bound":lower,"Upper Bound":upper,"Outlier Count":len(salary_outliers)}).round(2)

In [ ]:
plt.figure(figsize=(9,4))
sns.boxplot(data=analysis_df,x="Annual_Salary_USD")
plt.title("Salary Outlier Review")
plt.tight_layout()
plt.show()

Outliers are retained unless a documented data-quality reason supports removal.

## 11. Department Summary

In [ ]:
department_summary=(
    analysis_df.groupby("Department")
    .agg(
        Employees=("Employee_ID","count"),
        Average_Salary=("Annual_Salary_USD","mean"),
        Median_Salary=("Annual_Salary_USD","median"),
        Average_Performance=("Performance_Score","mean"),
        Average_Satisfaction=("Job_Satisfaction","mean"),
        Average_Training_Hours=("Training_Hours","mean"),
        Average_Absence_Days=("Absence_Days","mean"),
        Total_Projects=("Projects_Completed","sum")
    )
    .round(2)
    .sort_values("Average_Salary",ascending=False)
)
department_summary

## 12. Department Salary Visualization

In [ ]:
salary_by_department=analysis_df.groupby("Department")["Annual_Salary_USD"].mean().sort_values(ascending=False)
plt.figure(figsize=(9,5))
salary_by_department.plot(kind="bar")
plt.title("Average Salary by Department")
plt.ylabel("Average Salary (USD)")
plt.xticks(rotation=45,ha="right")
plt.tight_layout()
plt.show()

## 13. Performance by Department

In [ ]:
performance_by_department=analysis_df.groupby("Department")["Performance_Score"].mean().sort_values(ascending=False)
performance_by_department.round(2)

In [ ]:
plt.figure(figsize=(9,5))
performance_by_department.plot(kind="bar")
plt.title("Average Performance by Department")
plt.ylabel("Average Performance Score")
plt.xticks(rotation=45,ha="right")
plt.tight_layout()
plt.show()

## 14. Correlation Analysis

In [ ]:
corr_columns=[
    "Age","Years_Experience","Annual_Salary_USD","Weekly_Hours",
    "Performance_Score","Job_Satisfaction","Projects_Completed",
    "Training_Hours","Absence_Days"
]
corr=analysis_df[corr_columns].corr().round(2)
corr

In [ ]:
plt.figure(figsize=(11,8))
sns.heatmap(corr,annot=True,fmt=".2f",cmap="coolwarm",center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

## 15. Experience and Salary

In [ ]:
experience_salary_correlation=analysis_df["Years_Experience"].corr(analysis_df["Annual_Salary_USD"])
experience_salary_correlation

In [ ]:
plt.figure(figsize=(8,5))
sns.regplot(data=analysis_df,x="Years_Experience",y="Annual_Salary_USD",scatter_kws={"alpha":0.55})
plt.title("Experience and Annual Salary")
plt.tight_layout()
plt.show()

## 16. Training and Performance

In [ ]:
training_performance_correlation=analysis_df["Training_Hours"].corr(analysis_df["Performance_Score"])
training_performance_correlation

In [ ]:
plt.figure(figsize=(8,5))
sns.regplot(data=analysis_df,x="Training_Hours",y="Performance_Score",scatter_kws={"alpha":0.55})
plt.title("Training Hours and Performance")
plt.tight_layout()
plt.show()

Correlation does not establish causation.

## 17. Pivot Table: Salary by Department and Education

In [ ]:
salary_pivot=pd.pivot_table(
    analysis_df,
    values="Annual_Salary_USD",
    index="Department",
    columns="Education_Level",
    aggfunc="mean"
).round(2)
salary_pivot

## 18. Crosstab: Education and Department

In [ ]:
education_department_table=pd.crosstab(
    analysis_df["Education_Level"],
    analysis_df["Department"]
)
education_department_table

## 19. Hypothesis Test 1: Salary Between Two Gender Groups

In [ ]:
gender_values=analysis_df["Gender"].dropna().unique()
if len(gender_values)<2:
    raise ValueError("At least two gender groups are required.")
g1_name,g2_name=gender_values[:2]
g1=analysis_df.loc[analysis_df["Gender"]==g1_name,"Annual_Salary_USD"]
g2=analysis_df.loc[analysis_df["Gender"]==g2_name,"Annual_Salary_USD"]

pd.DataFrame({
    "Group":[g1_name,g2_name],
    "Count":[len(g1),len(g2)],
    "Mean Salary":[g1.mean(),g2.mean()],
    "Median Salary":[g1.median(),g2.median()]
}).round(2)

In [ ]:
levene=stats.levene(g1,g2)
equal_var=bool(levene.pvalue>=0.05)
gender_test=stats.ttest_ind(g1,g2,equal_var=equal_var)
pd.Series({
    "Levene p-value":levene.pvalue,
    "Equal Variance Assumed":equal_var,
    "t-statistic":gender_test.statistic,
    "p-value":gender_test.pvalue
}).round(4)

In [ ]:
def cohens_d(x,y):
    nx,ny=len(x),len(y)
    pooled=((nx-1)*x.var(ddof=1)+(ny-1)*y.var(ddof=1))/(nx+ny-2)
    return (x.mean()-y.mean())/np.sqrt(pooled)

gender_effect_size=cohens_d(g1,g2)
gender_effect_size

## 20. Hypothesis Test 2: Salary Across Departments

In [ ]:
department_groups=[g["Annual_Salary_USD"].values for _,g in analysis_df.groupby("Department")]
department_anova=stats.f_oneway(*department_groups)
pd.Series({"F-statistic":department_anova.statistic,"p-value":department_anova.pvalue}).round(4)

In [ ]:
grand_mean=analysis_df["Annual_Salary_USD"].mean()
ss_between=sum(len(g)*(g["Annual_Salary_USD"].mean()-grand_mean)**2 for _,g in analysis_df.groupby("Department"))
ss_total=((analysis_df["Annual_Salary_USD"]-grand_mean)**2).sum()
department_eta_squared=ss_between/ss_total
department_eta_squared

A significant ANOVA result shows that at least one department differs. Post-hoc testing is required to identify specific differences.

## 21. Hypothesis Test 3: Education and Department

In [ ]:
chi2,chi_p,chi_dof,expected=stats.chi2_contingency(education_department_table)
n=education_department_table.to_numpy().sum()
cramers_v=np.sqrt(chi2/(n*(min(education_department_table.shape)-1)))
pd.Series({
    "Chi-square":chi2,
    "Degrees of Freedom":chi_dof,
    "p-value":chi_p,
    "Cramer's V":cramers_v
}).round(4)

## 22. Integrated Findings Table

In [ ]:
findings=pd.DataFrame({
    "Question":[
        "Experience and salary relationship",
        "Training and performance relationship",
        "Salary difference between selected gender groups",
        "Salary difference across departments",
        "Education and department association"
    ],
    "Method":[
        "Pearson correlation","Pearson correlation","Independent t-test",
        "One-way ANOVA","Chi-square test"
    ],
    "Key_Result":[
        experience_salary_correlation,training_performance_correlation,
        gender_test.pvalue,department_anova.pvalue,chi_p
    ],
    "Effect_Size":[
        np.nan,np.nan,gender_effect_size,department_eta_squared,cramers_v
    ]
})
findings.round(4)

## 23. Interpretation Framework

For each result, state:

1. the analytical question;
2. the method;
3. the descriptive evidence;
4. statistical significance;
5. effect size;
6. practical meaning;
7. limitations.

## 24. Recommendations

Recommendations must follow directly from supported evidence. Appropriate examples include reviewing unusual departmental patterns, evaluating training alignment, and conducting more controlled compensation analyses.

## 25. Limitations

- The dataset is synthetic.
- The design is observational.
- Important explanatory variables may be missing.
- Group sizes may differ.
- Multiple tests increase false-positive risk.
- Artificial outliers are present.
- Results demonstrate methods rather than real organizational conclusions.

## 26. Export Final Outputs

In [ ]:
output_dir=Path("analysis_outputs")
output_dir.mkdir(exist_ok=True)

analysis_df.to_csv(output_dir/"cleaned_employee_statistics.csv",index=False)
department_summary.to_csv(output_dir/"department_summary.csv")
findings.to_csv(output_dir/"statistical_findings.csv",index=False)

sorted(p.name for p in output_dir.iterdir())

## 27. Student Project Tasks

1. Write an executive summary.
2. Identify three descriptive findings.
3. Explain two visualizations.
4. Interpret the strongest correlation.
5. Interpret one hypothesis test with its effect size.
6. Propose two evidence-based recommendations.
7. Identify three limitations.
8. Export the cleaned data and summary tables.

In [ ]:
# Write your final case-study conclusions here.

## Final Summary

This case study applied data validation, cleaning, feature creation, descriptive analysis, visualization, correlation, grouping, pivot tables, hypothesis testing, effect-size interpretation, reporting, and recommendations.

This completes the Data Science course.